# Open Insect ID v2 — data campaign (Kaggle)
#
# CPU notebook driving the 40k-species x 150-images campaign (architecture
# doc section 9): four resumable phases, one per run, each publishing the
# Kaggle dataset consumed by the next phase and by the training notebook
# (02):
#
#   PHASE="census"     page GBIF's speciesKey facet   -> meta dataset
#   PHASE="sample"     one occurrence page per species -> meta dataset
#   PHASE="download"   fill one shard (SHARD) with curated 320-px images
#                      -> data-NN + manifest datasets
#   PHASE="manifest"   merge the per-shard manifest parts -> manifest dataset
#
# PHASE / SHARD / RUN_BUDGET_MIN are injected by scripts/make_notebooks.py —
# the single source of truth (edit it, regenerate, push; never edit this
# .ipynb directly). Every phase skips artifacts that already exist and
# publishes from a `finally` block, so a killed or failed session simply
# continues on the next run without losing progress. GBIF 429 throttling
# is absorbed by AIMD pacing + patient retries (data_pipeline.campaign).
# Set the accelerator to **CPU**; only the training notebook (02) needs a
# **GPU**.


## 1. Setup & configuration

In [ ]:
import os, subprocess, sys
from pathlib import Path

# --- Configure this to your repo -------------------------------------------
GITHUB_REPO = os.environ.get("OPENINSECT_REPO", "https://github.com/YoannDev90/OpenInsectID.git")
# ---------------------------------------------------------------------------
IS_KAGGLE = os.path.exists("/kaggle/input")

def _is_our_checkout(path):
    # Identify OUR repo precisely: any folder happening to contain a src/
    # (cwd leftovers, /kaggle, ...) must not be mistaken for the checkout.
    return (os.path.isfile(os.path.join(path, "pyproject.toml"))
            and os.path.isdir(os.path.join(path, "src", "data_pipeline")))

# Walk up from cwd (kernels start in /kaggle/working, Jupyter in the notebook
# folder) to the repo root; anywhere else means: install from GitHub.
repo_root = os.getcwd()
while not _is_our_checkout(repo_root):
    parent = os.path.dirname(repo_root)
    if parent == repo_root:
        break
    repo_root = parent
in_checkout = _is_our_checkout(repo_root)

if in_checkout:
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
else:
    # pip needs the `git+` prefix to clone -- a plain https URL makes GitHub
    # serve the HTML page and pip fails with "cannot detect archive format".
    install_url = GITHUB_REPO
    if install_url.startswith(("http://", "https://")) and not install_url.startswith("git+"):
        install_url = "git+" + install_url
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", install_url])

try:
    import src.data_pipeline  # noqa: F401 - fail fast, with context, if shadowed
except ModuleNotFoundError:
    print("DIAG sys.path:", sys.path)
    raise

from pathlib import Path
import yaml

print(f"cwd={os.getcwd()} checkout={in_checkout} repo_root={repo_root}")
print("package ready; importable as `src.*`")


In [ ]:
# --- Run controls (injected by scripts/make_notebooks.py) -------------------
PHASE = 'sample'                  # census | sample | download | manifest
SHARD = 0                         # shard filled by the download phase (0..N_SHARDS-1)
RUN_BUDGET_MIN = 540               # wall-clock budget (minutes) for one sample/download run
N_SHARDS = 12                     # matches CampaignConfig + the data-NN datasets

import json, shutil, subprocess, sys
from pathlib import Path

import polars as pl

from src.data_pipeline import CampaignConfig

cfg = CampaignConfig(
    target_species=40_000,
    n_shards=N_SHARDS,
    # Section-9 defaults: min_occurrences=50, facet_page=1000, sample_limit=300,
    # part_species=500, max_images_per_species=150, image_size=320,
    # jpeg_quality=85, max_shard_bytes=18 GB (working-dir guard).
)
print(f"PHASE={PHASE}"
      + (f" SHARD={SHARD:02d}" if PHASE == "download" else "")
      + (f" budget={RUN_BUDGET_MIN} min" if PHASE in ("sample", "download") else ""))

# Both Kaggle mount layouts (classic + BYOD), same probe as notebooks 02/03.
MOUNTS = [Path("/kaggle/input"), Path("/kaggle/input/datasets/yoanndev90")]

def _input(slug):
    """Attached dataset root, or None when it is not mounted."""
    for base in MOUNTS:
        p = base / slug
        if p.exists():
            return p
    return None

def _publish(slug, folder, message, title):
    """Push `folder` as the next version of dataset `slug` (Kaggle runs only)."""
    folder = Path(folder)
    if not IS_KAGGLE:
        print(f"local run: {folder} not pushed -> {slug}")
        return
    # The preinstalled CLI (2.0.2) mangles license/keywords on version
    # pushes — upgrade it first (same guard as notebook 02).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"],
                   check=False)
    (folder / "dataset-metadata.json").write_text(json.dumps({
        "title": title,
        "id": slug,
        "licenses": [{"name": "Attribution 4.0 International (CC BY 4.0)"}],
        # Kaggle keywords are a closed tag list: unknown tags are dropped.
        "keywords": ["classification", "research", "python", "beginner"],
    }))
    # -r tar: directories upload uncompressed (images mount as plain files);
    # -t: never rewrite our parquet files to CSV on upload.
    r = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(folder),
         "-m", message, "-r", "tar", "-t", "-q"],
        capture_output=True, text=True,
    )
    print(f"push {slug} rc:", r.returncode, "|",
          (r.stdout + r.stderr).strip() or "ok")


## 2. `PHASE="census"` — which species are worth sampling

Page GBIF's `speciesKey` facet (Insecta, still images, PRESENT, human or
preserved records, has coordinate, `>= min_occurrences` eligible
occurrences) until a short page, then merge into `meta/census.parquet` plus
the deterministic `meta/shards.parquet` species→shard assignment (balanced
by estimated images, top `target_species`). Publishes the **meta** dataset.

Existing facet pages are skipped: a killed run resumes at the first missing
page. This phase attaches nothing.

In [ ]:
if PHASE != "census":
    print(f"skipped: PHASE={PHASE!r} (this cell runs the census)")
else:
    from src.data_pipeline import census_sync

    out = Path("meta")
    census_path = census_sync(cfg, out)   # stored pages are never refetched
    census = pl.read_parquet(census_path)
    plan = pl.read_parquet(out / "shards.parquet")
    print(f"census: {census.height} species with >= {cfg.min_occurrences} "
          f"eligible occurrences")
    print(f"shards: {plan.height} species -> {plan['shard'].n_unique()} shards, "
          f"est. {plan['est_images'].sum():,} images")
    _publish("yoanndev90/open-insect-id-v2-meta", out,
             f"census: {census.height} species", "Open Insect ID - Meta")


## 3. `PHASE="sample"` — one occurrence page per census species

Read `census.parquet` from the attached **meta** dataset and fetch up to
`sample_limit=300` eligible occurrences per species (lineage + image URLs),
500 species per `part-*.parquet` in census order. Existing parts are
skipped, so a killed run resumes at the first missing part; a species whose
request fails is retried once inside its part (GBIF throttling is normal).

Stops cleanly on `RUN_BUDGET_MIN` between parts, and the **meta** dataset is
republished from `finally` — every part written so far survives a failed
run, so progress is never lost.

In [ ]:
if PHASE != "sample":
    print(f"skipped: PHASE={PHASE!r} (this cell samples GBIF occurrences)")
else:
    from src.data_pipeline import sample_sync

    work = Path("meta")
    src = _input("open-insect-id-v2-meta")
    if src is not None:
        # Full snapshot: census + every part sampled so far.
        shutil.copytree(src, work, dirs_exist_ok=True)
    census_path = work / "census.parquet"
    if not census_path.exists():
        raise SystemExit("census.parquet not found: run PHASE=census first")
    try:
        parts = sample_sync(cfg, census_path, work / "sampling",  # skipped if present
                            budget_min=RUN_BUDGET_MIN)
    finally:
        # Publish on success AND on failure: a raised run still ships every
        # part it managed to write (the next run resumes from those).
        part_files = sorted((work / "sampling").glob("part-*.parquet"))
        n_rows = sum(pl.read_parquet(p).height for p in part_files)
        print(f"sample: {len(part_files)} parts / {n_rows} candidate rows")
        _publish("yoanndev90/open-insect-id-v2-meta", work,
                 f"sample: {len(part_files)} parts / {n_rows} rows",
                 "Open Insect ID - Meta")


## 4. `PHASE="download"` — fill one data shard

Materialise the previous version of `open-insect-id-v2-data-NN` into the
working directory (dataset versions are full snapshots), then download this
shard's species from the sampling index: fetch → curate (224-px floor,
aspect ≤ 3) → resize to 320 px → q85 JPEG, content-hash named under
`order/family/genus/species/`, capped at `max_images_per_species`.

Stops cleanly on `RUN_BUDGET_MIN` or the 18 GB working-dir guard; the next
run continues, skipping species that are already full without spending
network. The shard manifest + both pushes run from `finally`: even a
crashed run ships the images it downloaded (shard first, then this
shard's manifest part — a lagging part under-lists, which is safe).

In [ ]:
if PHASE != "download":
    print(f"skipped: PHASE={PHASE!r} (this cell fills data-{SHARD:02d})")
else:
    from src.data_pipeline import SamplingIndex, download_sync, write_shard_manifest

    # Read-only inputs: census + sampling index (attached meta, or local meta/).
    meta = _input("open-insect-id-v2-meta")
    if meta is None and (Path("meta") / "census.parquet").exists():
        meta = Path("meta")
    if meta is None:
        raise SystemExit("meta not found: attach open-insect-id-v2-meta "
                         "(or run census + sample locally first)")

    shard_id = f"open-insect-id-v2-data-{SHARD:02d}"
    work = Path(f"data-{SHARD:02d}")
    src = _input(shard_id)
    if src is not None:
        # Dataset versions are full snapshots: materialise the previous one so
        # this run continues from exactly the state the last run pushed.
        shutil.copytree(src, work, dirs_exist_ok=True)

    plan = pl.read_parquet(meta / "shards.parquet")
    keys = plan.filter(pl.col("shard") == SHARD)["species_key"].to_list()
    have = sum(p.stat().st_size for p in work.rglob("*.jpg")) if work.exists() else 0
    print(f"shard {SHARD:02d}: {len(keys)} species, {have / 1e9:.2f} GB on disk")

    index = SamplingIndex(meta / "census.parquet", meta / "sampling",
                          part_size=cfg.part_species)
    stats = {"ok": 0}  # so the finally below can report a crashed run
    try:
        stats = download_sync(cfg, work, keys, index, budget_min=RUN_BUDGET_MIN)
        print("download:", stats)
    finally:
        # Publish on success AND on failure: images already on disk are
        # never thrown away (a kill mid-push loses at most this run's tail).
        work.mkdir(parents=True, exist_ok=True)
        manifest = write_shard_manifest(work)
        rows = pl.read_parquet(manifest).height
        print(f"shard manifest: {rows} images -> {manifest}")

        # Shard first: a manifest part that lags behind under-lists images
        # (safe — the next run heals it); the reverse order could list
        # missing files.
        _publish(f"yoanndev90/{shard_id}", work,
                 f"shard {SHARD:02d}: +{stats['ok']} images ({rows} total)",
                 f"Open Insect ID - Data {SHARD:02d}")
        mwork = Path("manifest")
        msrc = _input("open-insect-id-v2-manifest")
        if msrc is not None:
            shutil.copytree(msrc, mwork, dirs_exist_ok=True)
        mwork.mkdir(exist_ok=True)
        shutil.copy(manifest, mwork / f"data-{SHARD:02d}.parquet")
        _publish("yoanndev90/open-insect-id-v2-manifest", mwork,
                 f"part data-{SHARD:02d}: {rows} images", "Open Insect ID - Manifest")


## 5. `PHASE="manifest"` — merge the shard manifests

Read every `data-NN.parquet` part from the attached **manifest** dataset,
merge into `insectid_manifest.parquet` with a `shard` column (the dataset
slug, for multi-root loading), slug-sorted so train/val splits stay
reproducible, and republish the **manifest** dataset. Run this once all
`N_SHARDS` shards report a manifest part.

In [ ]:
if PHASE != "manifest":
    print(f"skipped: PHASE={PHASE!r} (this cell merges the shard manifests)")
else:
    from src.data_pipeline import merge_manifests

    work = Path("manifest")
    src = _input("open-insect-id-v2-manifest")
    if src is not None:
        shutil.copytree(src, work, dirs_exist_ok=True)
    pairs, missing = [], []
    for i in range(N_SHARDS):
        name = f"data-{i:02d}"
        part = work / f"{name}.parquet"
        if part.exists():
            pairs.append((name, part))
        else:
            missing.append(name)
    if missing:
        raise SystemExit(f"missing manifest parts: {missing} — run the download "
                         f"phase for those shards first")
    df = merge_manifests(pairs, work / "insectid_manifest.parquet")
    print(f"merged {len(pairs)} shard parts -> {df.height} images, "
          f"{df['species'].n_unique()} species, {df['order'].n_unique()} orders")
    _publish("yoanndev90/open-insect-id-v2-manifest", work,
             f"merged: {df.height} images / {df['species'].n_unique()} species",
             "Open Insect ID - Manifest")
